<a href="https://colab.research.google.com/github/thaoS2/BI_Nhom7_BT/blob/main/Homework/CS06/notebook/02_regression_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from google.colab import files
from IPython.display import display


In [4]:
file_name = "/content/sp500_event_study_data.xlsx"

df = pd.read_excel(file_name)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

print("Số dòng dữ liệu giá:", len(df))
print("Khoảng thời gian:", df["Date"].min().date(), "→", df["Date"].max().date())

display(df.head())

Số dòng dữ liệu giá: 944
Khoảng thời gian: 2023-01-03 → 2026-10-07


,Date,Close,High,Low,Open,Volume,Return,Dt
0,2023-01-03,3824.139893,3878.459961,3794.330078,3853.290039,3959140000,NaN,0
1,2023-01-04,3852.969971,3873.159912,3815.770020,3840.360107,4414080000,0.753897,0
2,2023-01-05,3808.100098,3839.739990,3802.419922,3839.739990,3893450000,-1.164553,0
3,2023-01-06,3895.080078,3906.189941,3809.560059,3823.370117,3923560000,2.284078,0
4,2023-01-09,3892.090088,3950.570068,3890.419922,3910.820068,4311770000,-0.076763,0


In [5]:
df['Y_t'] = df['Close'].pct_change() * 100

if 'Return' in df.columns:
    max_diff = (df['Return'] - df['Y_t']).abs().max()
    print('Sai lệch lớn nhất giữa Return có sẵn và Y_t tính lại:', max_diff)

display(df[['Date', 'Close', 'Y_t', 'Dt']].head(10))

Sai lệch lớn nhất giữa Return có sẵn và Y_t tính lại: 8.881784197001252e-16


,Date,Close,Y_t,Dt
0,2023-01-03,3824.139893,NaN,0
1,2023-01-04,3852.969971,0.753897,0
2,2023-01-05,3808.100098,-1.164553,0
3,2023-01-06,3895.080078,2.284078,0
4,2023-01-09,3892.090088,-0.076763,0
5,2023-01-10,3919.250000,0.697823,0
6,2023-01-11,3969.610107,1.284942,0
7,2023-01-12,3983.169922,0.341591,0
8,2023-01-13,3999.090088,0.399686,0
9,2023-01-17,3990.969971,-0.203049,0


In [6]:
df['Dt'] = df['Dt'].fillna(0).astype(int)

print('Số ngày sự kiện trong file trước khi chỉnh:', int(df['Dt'].sum()))
display(df.loc[df['Dt'] == 1, ['Date', 'Close', 'Y_t', 'Dt']])

event_trade_date = pd.Timestamp('2026-01-26')
if event_trade_date in set(df['Date']):
    df.loc[df['Date'] == event_trade_date, 'Dt'] = 1

print('\nSố ngày sự kiện sau khi chỉnh:', int(df['Dt'].sum()))
event_days = df.loc[df['Dt'] == 1, ['Date', 'Close', 'Y_t', 'Dt']].copy()
display(event_days)

assert int(df['Dt'].sum()) == 5, 'Dt chưa có đúng 5 ngày sự kiện. Kiểm tra lại dữ liệu.'

Số ngày sự kiện trong file trước khi chỉnh: 4


,Date,Close,Y_t,Dt
192,2023-10-09,4335.660156,0.630385,1
325,2024-04-19,4967.229980,-0.875855,1
564,2025-04-03,5396.520020,-4.839563,1
791,2026-03-02,6881.620117,0.039835,1



Số ngày sự kiện sau khi chỉnh: 5


,Date,Close,Y_t,Dt
192,2023-10-09,4335.660156,0.630385,1
325,2024-04-19,4967.229980,-0.875855,1
564,2025-04-03,5396.520020,-4.839563,1
767,2026-01-26,6950.229980,0.500608,1
791,2026-03-02,6881.620117,0.039835,1


In [7]:
df['Y_t_lag1'] = df['Y_t'].shift(1)

model_df = df[['Date', 'Y_t', 'Y_t_lag1', 'Dt']].dropna().copy()

print('Số quan sát dùng trong hồi quy:', len(model_df))
print('Số ngày sự kiện trong mẫu hồi quy:', int(model_df['Dt'].sum()))
display(model_df.head())

Số quan sát dùng trong hồi quy: 942
Số ngày sự kiện trong mẫu hồi quy: 5


,Date,Y_t,Y_t_lag1,Dt
2,2023-01-05,-1.164553,0.753897,0
3,2023-01-06,2.284078,-1.164553,0
4,2023-01-09,-0.076763,2.284078,0
5,2023-01-10,0.697823,-0.076763,0
6,2023-01-11,1.284942,0.697823,0


In [9]:
# Y_t = beta_0 + beta_1*Y_t_lag1 + beta_2*Dt + epsilon_t
X = model_df[['Y_t_lag1', 'Dt']]
X = sm.add_constant(X)
y = model_df['Y_t']

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                    Y_t   R-squared:                       0.008
Model:                            OLS   Adj. R-squared:                  0.005
Method:                 Least Squares   F-statistic:                     3.576
Date:                Fri, 09 Oct 2026   Prob (F-statistic):             0.0284
Time:                        04:00:27   Log-Likelihood:                -1258.8
No. Observations:                 942   AIC:                             2524.
Df Residuals:                     939   BIC:                             2538.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0875      0.030      2.893      0.0

In [10]:
ci = model.conf_int(alpha=0.05)
coef_table = pd.DataFrame({
    'Coefficient': model.params,
    'Std_Error': model.bse,
    't_stat': model.tvalues,
    'p_value': model.pvalues,
    'CI_95_Lower': ci[0],
    'CI_95_Upper': ci[1]
})

display(coef_table)

b0 = model.params['const']
b1 = model.params['Y_t_lag1']
b2 = model.params['Dt']
print(f'Phương trình ước lượng: Ŷ_t = {b0:.6f} + ({b1:.6f})Y_(t-1) + ({b2:.6f})D_t')

,Coefficient,Std_Error,t_stat,p_value,CI_95_Lower,CI_95_Upper
const,0.087482,0.030236,2.893309,0.003900,0.028144,0.146819
Y_t_lag1,-0.038185,0.032506,-1.174705,0.240410,-0.101977,0.025608
Dt,-0.986989,0.413537,-2.386699,0.017198,-1.798553,-0.175425


Phương trình ước lượng: Ŷ_t = 0.087482 + (-0.038185)Y_(t-1) + (-0.986989)D_t


In [11]:
model_df['Predicted_Y_t'] = model.predict(X)
model_df['Residual'] = model_df['Y_t'] - model_df['Predicted_Y_t']

event_results = model_df.loc[model_df['Dt'] == 1,
    ['Date', 'Y_t', 'Y_t_lag1', 'Predicted_Y_t', 'Residual', 'Dt']].copy()

print('5 ngày sự kiện:')
display(event_results)

5 ngày sự kiện:


,Date,Y_t,Y_t_lag1,Predicted_Y_t,Residual,Dt
192,2023-10-09,0.630385,1.181489,-0.944622,1.575007,1
325,2024-04-19,-0.875855,-0.220816,-0.891075,0.015221,1
564,2025-04-03,-4.839563,0.672819,-0.925199,-3.914365,1
767,2026-01-26,0.500608,0.032687,-0.900755,1.401364,1
791,2026-03-02,0.039835,-0.433935,-0.882938,0.922773,1


In [12]:
model_df.to_csv('model_data_predictions_residuals.csv', index=False)
coef_table.to_csv('regression_results.csv')
event_results.to_csv('event_day_results.csv', index=False)

print('Đã tạo 3 file:')
print('- model_data_predictions_residuals.csv')
print('- regression_results.csv')
print('- event_day_results.csv')

Đã tạo 3 file:
- model_data_predictions_residuals.csv
- regression_results.csv
- event_day_results.csv
